# 04 · Gotowy model audio, bez treningu sieci
**Cel:** sprawdzić alternatywny opis dźwięku. AST zamienia uderzenie na 768 liczb. Używamy wag AudioSet w trybie zamrożonym. To model ogólnych zdarzeń audio, nie wyspecjalizowany detektor pęknięć.

Model wymaga jednorazowego pobrania przez `scripts/warm_model.py`. Ten notebook jest opcjonalny (priorytet 3). Obliczenia pozostają lokalne. CPU jest domyślny; 48 GB RAM wystarcza z dużym zapasem dla pojedynczych wywołań.

In [1]:
from pathlib import Path
import os, json, sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Otwórz cały folder projektu w VS Code."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from pinglab.audio import Config, load_audio, detect_hits, cut_hits, listen
from pinglab.session import load_session
SESSION = load_session(ROOT)
CONFIG, FMAX = SESSION.config, SESSION.fmax
MANIFEST, ANNOTATIONS = SESSION.manifest, SESSION.annotations
print("Dane:", SESSION.source_kind, "· manifest:", MANIFEST.relative_to(ROOT) if MANIFEST.is_relative_to(ROOT) else MANIFEST.name)
print("Wspólne pasmo do", FMAX, "Hz · konfiguracja: project.json")
if SESSION.source_kind == "synthetic":
    display(Markdown("**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**"))


Dane: synthetic · manifest: data/demo/manifest.csv
Wspólne pasmo do 20000.0 Hz · konfiguracja: project.json


**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**

In [2]:
# KONFIGURACJA
MANIFEST = SESSION.manifest
ANNOTATIONS = SESSION.annotations
OUT = SESSION.output / "ast"
OFFLINE = True  # przy nowej instalacji: najpierw python scripts/warm_model.py
from pinglab.dataset import analyze_manifest
events,quality = analyze_manifest(MANIFEST,OUT,CONFIG,FMAX,ANNOTATIONS if ANNOTATIONS.exists() else None)

## Oblicz embeddingi
Pierwszy przebieg liczy każde uderzenie; kolejne korzystają z cache zależnego od zawartości audio i wersji modelu. AST widzi sygnał przeskalowany do 16 kHz, więc nie wykorzystuje rezonansów powyżej 8 kHz. Krótkie okna są dopełniane zgodnie z ekstraktorem modelu; to niedopasowanie do treningowych klipów warto sprawdzić.

In [3]:
from pinglab.embeddings import ASTEncoder
import time
start = time.perf_counter()
encoder = ASTEncoder(ROOT/'models/ast',offline=OFFLINE,device='cpu')
encoded = encoder.add_to_events(events,ROOT/'data/cache/ast',CONFIG)
encoded.to_csv(OUT/'features_ast.csv',index=False)
print("Model:",encoder.revision,"· uderzeń:",len(encoded),"· sekund:",round(time.perf_counter()-start,1))

Model: f826b80d28226b62986cc218e5cec390b1096902 · uderzeń: 90 · sekund: 4.9


In [4]:
from pinglab.plots import embedding_map
if encoded.plate_id.nunique()>=3:
    display(embedding_map(encoded,'ast')[0])
    display(embedding_map(encoded,'ast',color='session_id')[0])

## Ten sam test, inna reprezentacja
Model AST pozostaje zamrożony. Uczymy wyłącznie mały klasyfikator na embeddingach. Normalizacja i klasyfikator są wewnątrz każdej iteracji testu; podział płytek jest taki sam jak w 03.

In [5]:
from pinglab.learning import evaluate
comparisons=[]
for family in ['resonance','physics','mfcc','ast']:
    try:
        metrics,pred,folds=evaluate(encoded,family,bootstrap=300)
        comparisons.append({k:metrics[k] for k in ['family','n_plates','balanced_accuracy','recall_nok','false_alarm_rate']})
        pred.to_csv(OUT/f'predictions_{family}.csv',index=False)
    except ValueError as error:
        print(family,error)
display(pd.DataFrame(comparisons))

,family,n_plates,balanced_accuracy,recall_nok,false_alarm_rate
0,resonance,30,0.975,1.0,0.05
1,physics,30,1.000,1.0,0.00
2,mfcc,30,1.000,1.0,0.00
3,ast,30,1.000,1.0,0.00


**Interpretacja:** jeśli klasyczne cechy są równie dobre, zostań przy prostszej metodzie. Jeśli AST grupuje głównie szum / sesję, nie ma powodu mu ufać. Wynik na symulacji nie wybiera metody dla klienta.

**Plan awaryjny:** zamrożone AST / PANNs / CLAP + mały klasyfikator; dopiero przy większej liczbie niezależnych detali rozważ odmrożenie końcowych warstw. Nie trenuj całej sieci na 30 płytkach. Źródła i ograniczenia: `docs/RESEARCH.md`.